# EDA и отладка расчёта батч-признаков

Вспомогательная тетрадь: знакомство с данными и проверка логики расчёта.

**Важно:** боевая логика лежит в `dags/calculate_batch_features.py`. Здесь мы её только вызываем и проверяем — дублировать код не нужно.

Параметры подключения берутся из сниппета `===Активация БД===` (не храните их в репозитории).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('../dags'))
import pandas as pd
import sqlalchemy
from calculate_batch_features import (
    load_source_tables, preprocess_tables, build_batch_features,
    FEATURE_COLUMNS, WINDOWS,
)

In [ ]:
# Параметры подключения (подставьте свои из сниппета активации БД)
PG = dict(
    host='YOUR_HOST', port=6432,
    dbname='YOUR_DB', user='YOUR_USER', password='YOUR_PASSWORD',
)
uri = (f"postgresql+psycopg2://{PG['user']}:{PG['password']}"
       f"@{PG['host']}:{PG['port']}/{PG['dbname']}?sslmode=require")
engine = sqlalchemy.create_engine(uri)

## 1. Знакомство с данными

In [ ]:
with engine.connect() as conn:
    tables = load_source_tables(conn)
for name, df in tables.items():
    print(name, df.shape)
    display(df.head(3))

In [ ]:
# Уникальные ключи, диапазоны дат, пропуски
tables['customers']['signup_date'] = pd.to_datetime(tables['customers']['signup_date'])
print('customers uniq:', tables['customers']['customer_id'].nunique())
print('sessions uniq cust/sess:', tables['sessions']['customer_id'].nunique(), tables['sessions']['session_id'].nunique())
print('events uniq evt:', tables['events']['event_id'].nunique())
print('orders uniq cust/order:', tables['orders']['customer_id'].nunique(), tables['orders']['order_id'].nunique())
print('event_type:\n', tables['events']['event_type'].value_counts())

## 2. Предобработка

Единый тип времени, дедуп событий по `event_id`, `total_usd` → число, привязка `customer_id` к событиям через `sessions`, отбрасывание невалидных дат.

In [ ]:
clean = preprocess_tables(tables)
for name, df in clean.items():
    print(name, df.shape)
print('events has customer_id:', 'customer_id' in clean['events'].columns)

## 3. Расчёт признаков на одном run_date

Выбираем `run_date = '2025-09-01'`. Признаки считаются только по данным строго до этой даты.

In [ ]:
run_date = '2025-09-01'
with engine.connect() as conn:
    features = build_batch_features(conn, run_date)
print('shape:', features.shape)
features.head()

## 4. Проверки

- одна строка на пару (customer_id, run_date);
- все признаки на месте;
- нет данных из будущего;
- корректная обработка пропусков и деления на ноль.

In [ ]:
assert features.duplicated(['customer_id','run_date']).sum() == 0, 'есть дубликаты ключа'
assert list(features.columns) == FEATURE_COLUMNS, 'не совпал набор/порядок колонок'
# Утечка: максимальная использованная дата события < run_date
ev = clean['events']
mask = (ev['timestamp'] >= pd.Timestamp(run_date) - pd.Timedelta(days=30)) & (ev['timestamp'] < pd.Timestamp(run_date))
print('max event ts in window:', ev[mask]['timestamp'].max(), '< run_date', run_date)
print('конверсии в [0,1]:', features[['cr_view_cart_30d','cr_cart_purchase_30d']].max().to_dict())
print('days_since min (>=-1):', features['days_since_last_purchase'].min())
features.describe().T[['mean','min','max']]

In [ ]:
# Пример результата для сверки с DAG
features.to_csv('sample_run_2025-09-01.csv', index=False)
print('saved sample_run_2025-09-01.csv')